# Parte 1 – Visualización con Plotly [6 pts]

## Paso 1: Carga de datos

In [143]:
import pandas as pd

df = pd.read_csv("datos/tabla_final.csv", dtype={"ubigeo": str}, encoding="utf-8-sig")

In [144]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ubigeo              25 non-null     object 
 1   departamento        25 non-null     object 
 2   capital             25 non-null     object 
 3   latitud             25 non-null     float64
 4   longitud            25 non-null     float64
 5   lluvia_total_mm     25 non-null     float64
 6   dias_lluvia_fuerte  25 non-null     int64  
 7   declaratorias       25 non-null     int64  
 8   prorrogas           25 non-null     int64  
dtypes: float64(3), int64(3), object(3)
memory usage: 1.9+ KB


## Paso 2: Tema y columnas

**Tema:** Lluvias y declaratorias de emergencia por departamento en el Perú, de enero a mayo de 2025.

**Columnas que usaremos:**

| Columna | Qué mide |
|---|---|
| `departamento` | Nombre del departamento (24 + Callao) |
| `lluvia_total_mm` | Lluvia acumulada en la capital del departamento (mm) |
| `dias_lluvia_fuerte` | Días con 20 mm o más de lluvia |
| `declaratorias` | Número de decretos supremos de estado de emergencia por lluvias |
| `prorrogas` | Número de prórrogas de esas declaratorias |

## Paso 3: Gráficos plotly

In [145]:
import plotly.express as px

### 1 Ranking de lluvias a nivel departamental

Generamos el ranking departamental por total de lluvias con una grafico de barras que ordena las variables de forma descendente. 

In [146]:
rank = df.sort_values("lluvia_total_mm", ascending=False)

fig = px.bar(rank.sort_values("lluvia_total_mm"),
             x="lluvia_total_mm",
             y="departamento",
             orientation="h",
             text="lluvia_total_mm",
             height=750)
# Número sin decimales y con separador de miles al final de cada barra
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside", cliponaxis=False)

# Estiramos el eje un 15 % para que el número de la barra más larga no se corte
fig.update_xaxes(range=[0, rank["lluvia_total_mm"].max() * 1.15])
fig.show()

### ¿Hay más declaratorias de emergencia en los departamentos más llueve?

Ahora, segun departamentos veremos las relaciones entre el total de lluvia y la cantidad de declaratorias, con un gráfico de dispersión.

In [147]:
# Columna solo para el tamaño: +1 para que los departamentos con 0 días no desaparezcan
df["tamano"] = df["dias_lluvia_fuerte"] + 1

fig2 = px.scatter(df,
                  x="lluvia_total_mm",
                  y="declaratorias",
                  hover_name="departamento",
                  size="tamano",
                  hover_data={"tamano": False, "dias_lluvia_fuerte": True},
                  labels={"dias_lluvia_fuerte": "Días con lluvia fuerte"},
                  height=650)

fig2.update_layout(margin=dict(t=100))
fig2.show()

### ¿Las lluvias afectan de forma diferente segun la región natural?

Para el tercer gráfico, la intención es poder profundizar en la idea de que la cantidad de lluvia no necesariamente indica algún peligro o desastre para la zona que lo enfrenta. Dividiendo en regiones naturales, podemos observar mejor cómo los departamentos de la selva (o donde predomina ese terreno) se ven afectadas y se delcaran en emergencia por valores de lluvia total que se concentran en valores altos, mientras que en departamentos de la costa se concentran en valores bajos.

In [148]:
# Región natural de cada departamento
regiones = {
    "Costa": ["Tumbes", "Piura", "Lambayeque", "La Libertad", "Lima", "Callao", "Ica", "Tacna"],
    "Sierra": ["Cajamarca", "Áncash", "Huánuco", "Pasco", "Junín", "Huancavelica",
               "Ayacucho", "Apurímac", "Cusco", "Puno", "Arequipa", "Moquegua"],
    "Selva": ["Amazonas", "San Martín", "Loreto", "Ucayali", "Madre de Dios"],
}
df["region"] = df["departamento"].map({d: r for r, deps in regiones.items() for d in deps})

# Columna solo para el tamaño: +1 para que los departamentos con 0 días no desaparezcan
df["tamano"] = df["dias_lluvia_fuerte"] + 1

fig3 = px.scatter(df,
                  x="lluvia_total_mm",
                  y="declaratorias",
                  facet_col="region",
                  facet_col_wrap=2,
                  facet_row_spacing=0.15,
                  color="region",
                  size="tamano",
                  hover_name="departamento",
                  hover_data={"tamano": False, "dias_lluvia_fuerte": True, "region": False},
                  category_orders={"region": ["Costa", "Sierra", "Selva"]},
                  labels={"dias_lluvia_fuerte": "Días con lluvia fuerte"},
                  height=850)

# Números del eje X también en los gráficos de arriba
fig3.update_xaxes(showticklabels=True)

# "region=Costa" → "Costa" en el título de cada panel
fig3.for_each_annotation(lambda a: a.update(text=a.text.split("=")[1]))
fig3.update_layout(margin=dict(t=130), showlegend=False)
# Eje Y de 0 a 10, de dos en dos
fig3.update_yaxes(range=[0, 10], dtick=2)
fig3.show()

## Paso 4: Verificación rápida

Comprobamos que los valores más altos y más bajos de la tabla coinciden con lo que muestran los gráficos.

### Gráfico 1 y 2

In [149]:
cols = ["departamento", "region", "lluvia_total_mm", "declaratorias", "dias_lluvia_fuerte"]

# Gráfico 1 (barras): los 3 con más lluvia y los 3 con menos
print("Más lluvia:")
display(df.nlargest(3, "lluvia_total_mm")[cols])
print("Menos lluvia:")
display(df.nsmallest(3, "lluvia_total_mm")[cols])

# Gráficos 2 y 3 (dispersión): extremos de declaratorias
print("Más declaratorias:")
display(df.nlargest(3, "declaratorias")[cols])
print("Menos declaratorias:")
display(df.nsmallest(3, "declaratorias")[cols])

Más lluvia:


,departamento,region,lluvia_total_mm,declaratorias,dias_lluvia_fuerte
15,Loreto,Selva,2019.9,8,28
16,Madre de Dios,Selva,1751.3,3,30
1,Áncash,Sierra,1415.9,8,11


Menos lluvia:


,departamento,region,lluvia_total_mm,declaratorias,dias_lluvia_fuerte
22,Tacna,Costa,17.4,6,0
6,Callao,Costa,54.0,2,0
14,Lima,Costa,54.0,8,0


Más declaratorias:


,departamento,region,lluvia_total_mm,declaratorias,dias_lluvia_fuerte
7,Cusco,Sierra,801.3,9,4
0,Amazonas,Selva,427.9,8,0
1,Áncash,Sierra,1415.9,8,11


Menos declaratorias:


,departamento,region,lluvia_total_mm,declaratorias,dias_lluvia_fuerte
6,Callao,Costa,54.0,2,0
23,Tumbes,Costa,699.1,2,7
13,Lambayeque,Costa,128.1,3,0


## Paso 5: Ajustamos titulos y nombres de ejes

##### Ranking de lluvias a nivel departamental

In [150]:
# Título
# Título
fig.update_layout(title="Loreto y Madre de Dios recibieron más de 100 veces la lluvia de Tacna <br>(Enero–Mayo 2025)",
                  margin=dict(t=120))

# Nombres de los ejes
fig.update_xaxes(title_text="Lluvia total (mm)<br><sup>Fuente: Open-Meteo, API histórica.</sup>")
fig.update_yaxes(title_text="Departamento")
fig.show()

##### ¿Hay más declaratorias de emergencia en los departamentos más llueve?

In [151]:
# Título
fig2.update_layout(title="Más lluvia no siempre significó más declaratorias de emergencia<br>(Enero–Mayo 2025)"
                         "<br><sup>El tamaño del punto indica los días con lluvia fuerte (20 mm o más)</sup>")

# Nombres de los ejes
fig2.update_xaxes(title_text="Lluvia total (mm)<br><br><sup>Fuente: Open-Meteo (lluvia) y PCM, gob.pe (declaratorias). Elaboración propia.</sup>")
fig2.update_yaxes(title_text="Declaratorias de emergencia")
fig2.show()

##### ¿Las lluvias afectan de forma diferente segun la región natural?

In [152]:
# Título
fig3.update_layout(title="La sierra tuvo más declaratorias que la selva, aunque llovió menos <br>(Enero–Mayo 2025)"
                         "<br><sup>El tamaño del punto indica los días con lluvia fuerte (20 mm o más)</sup>",
                   margin=dict(b=120))

# Nombres de los ejes
fig3.update_xaxes(title_text="Lluvia total (mm)")
fig3.update_yaxes(title_text="Declaratorias de emergencia", col=1)

# Fuente una sola vez, al pie de la figura
fig3.add_annotation(text="Fuente: Open-Meteo (lluvia) y PCM, gob.pe (declaratorias). Elaboración propia.",
                    xref="paper", yref="paper", x=0.5, y=-0.12,
                    showarrow=False, xanchor="center", font=dict(size=11, color="gray"))

## Paso 6: Exportar a HTML

In [153]:
fig3.write_html("grafico.html", include_plotlyjs="cdn")
print("Gráfico exportado a grafico.html")

Gráfico exportado a grafico.html
